# 01 · Generate & land raw source-system extracts (synthetic)

**Altura Bonding & Tapes · NPI Launch Readiness Control Tower**

Emulates the nightly extracts that land from Altura's systems of record into a Unity Catalog
**landing volume**, which Lakeflow (Auto Loader) then ingests incrementally:

| Source system | Extract | Format |
|---|---|---|
| SAP | `MARA` material master, `LFA1` vendors, `ZQM_SRC_QUAL` supplier qualification / approved sources, `STPO` BOM, `EKPO` open POs | CSV (SAP field names) |
| MES | coating-line batch runs (pilot + production) | JSON lines |
| PLM | NPI project register, stage-gate events | JSON lines |
| OEM EDI | DELFOR-style weekly program demand | CSV |
| Mfg Eng | coating line capacity | CSV |
| Commercial | historical launch post-mortems | CSV |

* `batch=1` → full extract as of **2026-11-16**
* `batch=2` → **this morning's (2026-11-23) incremental drop**: two MES production-trial runs for
  the AFT-7200 launch and a supplier capacity-allocation notice from Arden Specialty Polymers.

All names (products, suppliers, OEMs, plants) are fictional. No real customer data is used.

In [1]:
import io, os, sys
sys.path.insert(0, os.path.abspath("../src"))
from npi.env import CATALOG, SCHEMA, LANDING, get_spark, get_ws, widget
from npi.datagen import generate, BATCH1_DATE, BATCH2_DATE

BATCH = widget("batch", "all")      # "1", "2" or "all"
RESET = widget("reset", "true") == "true"
spark, w = get_spark(), get_ws()
print(f"Target: {CATALOG}.{SCHEMA}  landing volume: {LANDING}  batch={BATCH} reset={RESET}")

/Users/anders.jensen/projects/altura-npi-launch-readiness/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


/Users/anders.jensen/projects/altura-npi-launch-readiness/.venv/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:147: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Target: andersjensen_fevm_3_catalog.npi_launch_readiness  landing volume: /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing  batch=all reset=True


In [2]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA} COMMENT "
          "'Altura BTD NPI Launch Readiness Control Tower - synthetic data (FE Bar build)'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.landing COMMENT "
          "'Raw landing zone for SAP / MES / PLM / OEM EDI extracts (Auto Loader source)'")
def _rm(path):
    for e in w.files.list_directory_contents(path):
        _rm(e.path) if e.is_directory else w.files.delete(e.path)
    w.files.delete_directory(path)

if RESET and BATCH in ("1", "all"):
    # wipe landing zone so the demo is reproducible from scratch
    for d in w.files.list_directory_contents(LANDING):
        if d.is_directory:
            _rm(d.path)
    print("Landing zone reset.")

Landing zone reset.


In [3]:
raw = generate()
JSONL = {"mes/batch_runs", "plm/npi_projects", "plm/stage_gates"}

def land(tables: dict, as_of):
    tag = as_of.strftime("%Y%m%d")
    out = []
    for key, df in tables.items():
        name = key.split("/")[-1]
        if key in JSONL:
            body, ext = df.to_json(orient="records", lines=True, date_format="iso"), "jsonl"
        else:
            body, ext = df.to_csv(index=False), "csv"
        path = f"{LANDING}/{key}/{name}_{tag}.{ext}"
        w.files.upload(path, io.BytesIO(body.encode()), overwrite=True)
        out.append((key, len(df), path))
    return out

landed = []
if BATCH in ("1", "all"):
    landed += land(raw.batch1, BATCH1_DATE)
if BATCH in ("2", "all"):
    landed += land(raw.batch2, BATCH2_DATE)
for k, n, p in landed:
    print(f"{k:28s} {n:>8,} rows -> {p}")

sap/MARA                          272 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/sap/MARA/MARA_20261116.csv
sap/LFA1                           55 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/sap/LFA1/LFA1_20261116.csv
sap/ZQM_SRC_QUAL                  489 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/sap/ZQM_SRC_QUAL/ZQM_SRC_QUAL_20261116.csv
sap/STPO                        2,393 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/sap/STPO/STPO_20261116.csv
sap/EKPO                          275 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/sap/EKPO/EKPO_20261116.csv
mes/batch_runs                 73,309 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/mes/batch_runs/batch_runs_20261116.jsonl
plm/npi_projects                  456 rows -> /Volumes/andersjensen_fevm_3_catalog/npi_launch_readiness/landing/plm/npi_projects/npi_pro

### Sanity check: the story is in the raw data
Historical launch outcomes (what the commercial team's post-mortems recorded) and the hero launch.

In [4]:
o = raw.batch1["commercial/launch_outcomes"]
failed = (o.slip_weeks > 2) | (o.otif_first_90d < 0.85)
print(f"Historical launches 2020-2026: {len(o)}")
print(f"  launches that missed ramp (slip>2w or 90-day OTIF<85%): {failed.mean():.1%}")
print(f"  OEM platforms lost: {o.platform_lost.sum()}  (lifetime value ${o.lost_platform_value_usd.sum()/1e6:,.0f}M)")
print(f"  expedite cost ${o.expedite_cost_usd.sum()/1e6:,.1f}M | E&O ${o.eo_writeoff_usd.sum()/1e6:,.1f}M | "
      f"deferred revenue ${o.deferred_revenue_usd.sum()/1e6:,.1f}M")
print(o[failed].root_cause_primary.value_counts().to_string())

Historical launches 2020-2026: 420
  launches that missed ramp (slip>2w or 90-day OTIF<85%): 31.9%
  OEM platforms lost: 22  (lifetime value $471M)
  expedite cost $87.4M | E&O $27.1M | deferred revenue $132.1M
root_cause_primary
Production yield below plan (pilot-to-line gap)    54
Component qualification (PPAP) late vs SOP         50
OEM schedule change                                13
Coating line capacity conflict                      9
Supplier capacity shortfall at ramp                 8


In [5]:
p = raw.batch1["plm/npi_projects"]
print(p[p.npi_id == "NPI-2027-014"].T.to_string())

                                                                        433
npi_id                                                         NPI-2027-014
product_code                                                       AFT-7200
product_name                AFT-7200 Structural Battery-Enclosure Foam Tape
family_code                                                             SAF
family_name                                    Structural Acrylic Foam Tape
business_unit                                               Bonding & Tapes
market_segment                                                       EV_OEM
oem_customer                                               Northstar Motors
oem_program                                                        NSM-EVX9
plant                                                                  US20
production_line                                                    US20-F02
pilot_line                                                        US10-PL01
sop_date    